In [1]:
import os
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from pydantic import BaseModel
from typing import Literal, Optional

In [3]:
#printing path of current directory
print(os.getcwd())

/Users/dikshitrishi/WorkWork/emotion-voice-agent


In [4]:
# check for asr where input will be audio and output will be text

import whisper
from backend.config import WHISPER_MODEL
model = whisper.load_model(WHISPER_MODEL)

PATH_TO_AUDIO = "sample.wav"

result = model.transcribe(PATH_TO_AUDIO, fp16=False, language="en")
print(result["text"])


 It's very hot here and I'm feeling very sad because the weather is not good.


In [5]:
print(type(result["text"]))

<class 'str'>


In [6]:
from groq import Groq
import json
from backend.config import GROQ_API_KEY, GROQ_MODEL

client = Groq(api_key=GROQ_API_KEY)

SYSTEM_PROMPT = """You are a warm, emotionally intelligent voice assistant.

For EVERY response you must reply with valid JSON in this exact format:
{
  "text": "<your spoken response here>",
  "emotion": "<one of: happy, sad, neutral, empathetic, excited, calm>"
}

Rules:
- "text" should sound natural when spoken aloud. No markdown, no bullet points.
- "emotion" must reflect the tone of YOUR response (not the user's mood).
- Keep responses concise (2-4 sentences max) — this is a voice interface.
- Be warm, helpful, and human-sounding."""

def chat(user_text: str, history: list[dict]) -> dict:
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    messages += history
    messages.append({"role": "user", "content": user_text})

    response = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=messages,
        response_format={"type": "json_object"},
        temperature=0.7,
        max_tokens=300,
    )

    raw = response.choices[0].message.content
    parsed = json.loads(raw)

    if "text" not in parsed or "emotion" not in parsed:
        raise ValueError(f"LLM returned unexpected schema: {raw}")

    return parsed

In [10]:
llm_output=chat(result["text"], [])
type(llm_output)

dict

In [ ]:
# checking the tts part
import base64
import io
import numpy as np
import soundfile as sf
from TTS.api import TTS
from backend.config import COQUI_MODEL, EMOTION_STYLE_MAP

# Load once at import time — first run downloads the model (~150MB)
_tts = None

def get_tts():
    global _tts
    if _tts is None:
        print("[TTS] Loading Coqui model...")
        _tts = TTS(model_name=COQUI_MODEL, progress_bar=False)
    return _tts

def synthesize(text: str, emotion: str) -> bytes:
    """
    Synthesize speech and apply speed/energy adjustments per emotion.
    Returns WAV audio as bytes.
    """
    tts = get_tts()
    style = EMOTION_STYLE_MAP.get(emotion, EMOTION_STYLE_MAP["neutral"])

    # Coqui writes to a file or returns numpy — we use a BytesIO buffer
    buf = io.BytesIO()
    tts.tts_to_file(text=text, file_path="/tmp/tts_out.wav")

    # Load the generated audio
    audio, sample_rate = sf.read("/tmp/tts_out.wav")

    # Apply speed shift via resampling
    speed = style["speed"]
    if speed != 1.0:
        import librosa
        audio = librosa.effects.time_stretch(audio.astype(np.float32), rate=speed)

    # Apply energy (volume) scaling
    audio = audio * style["energy"]
    audio = np.clip(audio, -1.0, 1.0)  # prevent clipping distortion

    # Write final audio to buffer
    sf.write(buf, audio, sample_rate, format="WAV")
    buf.seek(0)
    return buf.read()


def audio_to_base64(audio_bytes: bytes) -> str:
    return base64.b64encode(audio_bytes).decode("utf-8")


/Applications/miniconda3/envs/voice/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Applications/miniconda3/envs/voice/lib/python3.11/site-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [18]:
audio_bytes = synthesize(llm_output["text"], "happy")

 > Text splitted to sentences.
["I'm so sorry to hear that the weather is getting you down.", "Sometimes the heat can be overwhelming and it's totally normal to feel sad when the weather isn't what we want it to be.", 'Would you like some suggestions on how to stay cool and maybe find some indoor activities to lift your mood?']
 > Processing time: 5.2242820262908936
 > Real-time factor: 0.28226422815235575


In [19]:
from IPython.display import Audio
Audio(audio_bytes, rate=22050)